In [ ]:
# AGI Bench: Learning Curves
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Cognitive Science Rationale

Human learning follows a **power-law curve** — rapid initial improvement that decelerates (Newell & Rosenbloom, 1981). This benchmark measures in-context learning dynamics across 5 exposure levels (0, 2, 4, 8, 12 examples) using procedurally generated rule systems.


## Interpreting the Score

Combines power-law fit quality and learning rate. Higher scores indicate faster, more human-like learning trajectories.


### References
Newell & Rosenbloom (1981), Anderson (1982)


# Learning Curves Benchmark

Tests how performance improves with increasing training examples.
Uses procedurally generated rule systems.

**Cognitive Science**: Power Law of Practice (Newell & Rosenbloom, 1981)
**Key innovation**: Novel rule systems that cannot be in training data

In [ ]:
"""
Novel Rule System Generator for Learning Benchmarks.

Generates procedural rule systems that cannot be in training data.
Each system defines a mapping from inputs to outputs via a chain
of deterministic rules. Difficulty is controlled by:
- Number of rules
- Number of input features
- Rule interaction complexity (independent vs. chained)

Systems are seeded for reproducibility across runs.
"""

import random
import hashlib
from dataclasses import dataclass, field


@dataclass
class RuleSystem:
    """A generated rule system with examples."""
    name: str
    description: str
    rules: list[str]
    examples: list[dict]  # {"input": str, "output": str}
    test_items: list[dict]  # {"input": str, "output": str}
    difficulty: int  # 1-3
    n_rules: int
    domain: str  # "symbol", "language", "number"


def _make_rng(seed: str) -> random.Random:
    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)
    return random.Random(h)


def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a symbol transformation rule system.

    Input: sequence of symbols (e.g., "△ ○ □")
    Rules: transformations (e.g., "△ followed by ○ becomes ★")
    Output: transformed sequence
    """
    rng = _make_rng(seed)

    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]
    colors = ["red", "blue", "green", "yellow"]

    if difficulty == 1:
        # Simple 1-to-1 substitution
        src = rng.sample(shapes[:4], 3)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]
        mapping = dict(zip(src, dst[:3]))
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            return [mapping.get(s, s) for s in seq]

    elif difficulty == 2:
        # Context-dependent: pairs matter
        src = rng.sample(shapes[:5], 4)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]
        mapping = dict(zip(src[:3], dst[:3]))
        pair_rule = (src[0], src[1], dst[3])  # "X followed by Y becomes Z"
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} → both become {pair_rule[2]}")
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            result = []
            i = 0
            while i < len(seq):
                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:
                    result.extend([pair_rule[2], pair_rule[2]])
                    i += 2
                else:
                    result.append(mapping.get(seq[i], seq[i]))
                    i += 1
            return result

    else:  # difficulty == 3
        # Multi-pass with conditional rules
        src = rng.sample(shapes[:6], 5)
        dst = rng.sample(shapes, 5)
        mapping1 = {src[0]: dst[0], src[1]: dst[1]}
        mapping2 = {dst[0]: dst[2]}  # Chain: src[0] → dst[0] → dst[2]
        cond = src[2]  # If this symbol is present, apply extra rule
        extra_map = {src[3]: dst[3]}

        rules = [
            f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()
        ]
        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")
        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")
        rules.append("All other symbols stay the same throughout")

        def apply_rules(seq):
            # Pass 1
            result = [mapping1.get(s, s) for s in seq]
            # Pass 2
            result = [mapping2.get(s, s) for s in result]
            # Conditional
            if cond in seq:  # Check original sequence
                result = [extra_map.get(s, s) for s in result]
            return result

    # Generate examples
    all_items = []
    for _ in range(25):
        length = rng.randint(3, 6)
        seq = [rng.choice(shapes[:5]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    # Deduplicate by input
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_examples = min(15, len(unique_items) - 5)
    examples = unique_items[:n_examples]
    test_items = unique_items[n_examples:n_examples + 5]

    return RuleSystem(
        name=f"SymbolTransform-{seed}",
        description="Apply symbol transformation rules to input sequences",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="symbol",
    )


def generate_number_system(seed: str = "num_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a novel number system / arithmetic.

    Input: expression in the invented system
    Rules: how operators work
    Output: numeric result
    """
    rng = _make_rng(seed)

    op_names = ["grok", "flim", "zorp", "quex", "blix"]
    ops = rng.sample(op_names, 3)

    if difficulty == 1:
        # Two operators: basic arithmetic with twist
        a_op, b_op = ops[0], ops[1]
        a_fn = lambda x, y: x + y + 1  # "grok" = add and increment
        b_fn = lambda x, y: abs(x - y)  # "flim" = absolute difference
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: absolute difference of x and y",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    elif difficulty == 2:
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        a_fn = lambda x, y: x * 2 + y
        b_fn = lambda x, y: (x + y) % 10
        c_fn = lambda x, y: max(x, y) - min(x, y) + 1
        rules = [
            f"'{a_op}(x, y)' means: double x, then add y",
            f"'{b_op}(x, y)' means: add x and y, take the last digit (mod 10)",
            f"'{c_op}(x, y)' means: difference of larger and smaller, plus 1",
        ]
        op_map = {a_op: a_fn, b_op: b_fn, c_op: c_fn}

    else:  # difficulty == 3
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        # Nested operations
        a_fn = lambda x, y: x + y + 1
        b_fn = lambda x, y: x * y
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: multiply x and y",
            f"Operations can be nested: '{a_op}({b_op}(x, y), z)' means: first compute {b_op}(x, y), then use the result as the first argument to {a_op}",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    # Generate examples
    all_items = []
    for _ in range(20):
        if difficulty <= 2:
            op_name = rng.choice(list(op_map.keys()))
            x = rng.randint(1, 9)
            y = rng.randint(1, 9)
            result = op_map[op_name](x, y)
            expr = f"{op_name}({x}, {y})"
        else:
            # Allow nesting
            if rng.random() < 0.5:
                op_name = rng.choice(list(op_map.keys()))
                x = rng.randint(1, 9)
                y = rng.randint(1, 9)
                result = op_map[op_name](x, y)
                expr = f"{op_name}({x}, {y})"
            else:
                inner_op = rng.choice(list(op_map.keys()))
                outer_op = rng.choice(list(op_map.keys()))
                x, y, z = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)
                inner_result = op_map[inner_op](x, y)
                result = op_map[outer_op](inner_result, z)
                expr = f"{outer_op}({inner_op}({x}, {y}), {z})"

        all_items.append({"input": expr, "output": str(result)})

    # Deduplicate
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_ex = min(12, len(unique_items) - 5)
    examples = unique_items[:n_ex]
    test_items = unique_items[n_ex:n_ex + 5]

    return RuleSystem(
        name=f"NumberSystem-{seed}",
        description="Evaluate expressions using novel arithmetic operators",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="number",
    )


# Pre-generated systems for the benchmark
LEARNING_CURVE_SYSTEMS = [
    generate_symbol_system("lc_sym_easy", difficulty=1),
    generate_symbol_system("lc_sym_med", difficulty=2),
    generate_symbol_system("lc_sym_hard", difficulty=3),
    generate_number_system("lc_num_easy", difficulty=1),
    generate_number_system("lc_num_med", difficulty=2),
    generate_number_system("lc_num_hard", difficulty=3),
    generate_symbol_system("lc_sym_extreme1", difficulty=3),
    generate_number_system("lc_num_extreme2", difficulty=3),
]

# Systems for transfer testing
TRANSFER_BASE_SYSTEM = generate_symbol_system("transfer_base", difficulty=2)
TRANSFER_NEAR_SYSTEM = generate_symbol_system("transfer_near", difficulty=2)
TRANSFER_FAR_SYSTEM = generate_number_system("transfer_far", difficulty=2)

# Systems for interference testing
INTERFERENCE_A = generate_symbol_system("interf_a", difficulty=2)
INTERFERENCE_B = generate_symbol_system("interf_b_similar", difficulty=2)


# ── Abstract rule systems for far-transfer ──────────────────────────
def generate_abstract_system(seed: str, base_system: RuleSystem) -> RuleSystem:
    """Generate a far-transfer system: same abstract rules, different surface features."""
    rng = _make_rng(seed)
    if base_system.domain == "symbol":
        animals = ["cat", "dog", "fish", "bird", "frog", "ant", "bee", "owl"]
        shapes_used = sorted({s for rule in base_system.rules for s in ["\u25b3","\u25cb","\u25a1","\u25c7","\u2605","\u2b21","\u2b1f","\u25bd"] if s in rule})
        animal_pool = rng.sample(animals, min(len(shapes_used)+2, len(animals)))
        shape_map = {s: animal_pool[i % len(animal_pool)] for i, s in enumerate(shapes_used)}
        def remap(text):
            for s, a in shape_map.items():
                text = text.replace(s, a)
            return text
        return RuleSystem(
            name=f"AbstractTransfer-{seed}",
            description="Apply transformation rules to animal-name sequences (same structure, different surface)",
            rules=[remap(r) for r in base_system.rules],
            examples=[{"input": remap(e["input"]), "output": remap(e["output"])} for e in base_system.examples],
            test_items=[{"input": remap(t["input"]), "output": remap(t["output"])} for t in base_system.test_items],
            difficulty=base_system.difficulty + 1,
            n_rules=len(base_system.rules),
            domain="abstract",
        )
    else:
        new_rules = [r.replace("add","combine").replace("multiply","merge").replace("double","twin") for r in base_system.rules]
        return RuleSystem(
            name=f"AbstractTransfer-{seed}",
            description="Evaluate expressions using renamed operators (same math, different names)",
            rules=new_rules,
            examples=base_system.examples,
            test_items=base_system.test_items,
            difficulty=base_system.difficulty + 1,
            n_rules=len(new_rules),
            domain="abstract",
        )


FAR_TRANSFER_PAIRS = [
    {"base": generate_symbol_system("ft_sym_1", difficulty=2), "transfer": None},
    {"base": generate_symbol_system("ft_sym_2", difficulty=3), "transfer": None},
    {"base": generate_number_system("ft_num_1", difficulty=2), "transfer": None},
    {"base": generate_number_system("ft_num_2", difficulty=3), "transfer": None},
]
for pair in FAR_TRANSFER_PAIRS:
    pair["transfer"] = generate_abstract_system(f"xfer_{pair['base'].name}", pair["base"])

# Hard condition systems — reduced training window (only 3 examples)
HARD_LEARNING_SYSTEMS = [
    generate_symbol_system("lc_hard_sym_steep", difficulty=3),
    generate_number_system("lc_hard_num_steep", difficulty=3),
    generate_symbol_system("lc_hard_abstract_1", difficulty=3),
    generate_number_system("lc_hard_abstract_2", difficulty=3),
]


In [ ]:
"""
Learning Benchmark 1: Novel Rule System Learning Curves (v2)

Measures how model performance improves with increasing training examples,
including far-transfer and steep learning conditions.

Three conditions:
  A) Standard learning curves (0.25 weight) — original 8 systems, checkpoints 0-12
  B) Far-transfer (0.50 weight) — train on base system, test on abstract reskin
  C) Steep/hard (0.25 weight) — only 3 training examples before test on difficulty-3 systems

Cognitive Science Basis:
- Power Law of Practice (Newell & Rosenbloom, 1981)
- Transfer of learning (Thorndike & Woodworth, 1901)
- Sample efficiency as a measure of learning ability

Score: 0.25*standard + 0.50*far_transfer + 0.25*steep
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
import numpy as np
import re
import json
# data already loaded from previous cell


@dataclass
class RuleAnswer:
    answer: str
    reasoning: str


CHECKPOINTS = [0, 2, 4, 8, 12]
HARD_CHECKPOINTS = [0, 3]  # Only 0 and 3 examples for steep condition


def normalize_output(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text


def check_output(model_output: str, expected: str) -> bool:
    m = normalize_output(model_output)
    e = normalize_output(expected)
    return e in m or m in e


def _eval_system(llm, system, n_examples, test_items, chat_label):
    """Evaluate a system with n_examples training, return accuracy on test_items."""
    n_actual = min(n_examples, len(system.examples))
    with kbench.chats.new(chat_label):
        prompt_parts = [
            f"You are learning the rule system: **{system.name}**\n",
            f"Description: {system.description}\n",
            "\n**Rules:**",
        ]
        for rule in system.rules:
            prompt_parts.append(f"- {rule}")

        if n_actual > 0:
            prompt_parts.append(f"\n**Training examples ({n_actual}):**")
            for ex in system.examples[:n_actual]:
                prompt_parts.append(f"  Input: {ex['input']}  →  Output: {ex['output']}")

        n_correct = 0
        for test_item in test_items:
            test_prompt = "\n".join(prompt_parts) + (
                f"\n\nNow apply the rules to this new input:\n"
                f"Input: {test_item['input']}\n\n"
                f"Respond with ONLY a JSON object:\n"
                f'{{"answer": "<output after applying rules>", "reasoning": "<your steps>"}}'
            )
            try:
                result = llm.prompt(test_prompt, schema=RuleAnswer)
                answer = result.answer
            except Exception:
                raw = llm.prompt(test_prompt)
                try:
                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
                    answer = str(parsed.get("answer", raw))
                except Exception:
                    answer = raw
            if check_output(answer, test_item["output"]):
                n_correct += 1

        return n_correct / len(test_items) if test_items else 0


@kbench.task(name="Learning Curves")
def learning_curves(llm) -> float:
    """
    Learning Curves Benchmark v2.

    Score = 0.25*standard + 0.50*far_transfer + 0.25*steep
    """

    # ═══ CONDITION A: Standard Learning Curves (0.25) ═══
    all_curves = []
    for system in LEARNING_CURVE_SYSTEMS:
        curve = {"system": system.name, "difficulty": system.difficulty, "checkpoints": []}
        for n_ex in CHECKPOINTS:
            acc = _eval_system(llm, system, n_ex, system.test_items,
                               f"{system.name}_n{n_ex}")
            curve["checkpoints"].append({"n_examples": n_ex, "accuracy": acc})
        all_curves.append(curve)

    # Compute standard score (same as v1)
    asymptotic_accs = []
    learning_rates = []
    sample_efficiencies = []
    curve_qualities = []

    for curve in all_curves:
        y = np.array([c["accuracy"] for c in curve["checkpoints"]])
        asymptotic_accs.append(float(y[-1]))
        learning_rates.append(max(0, float(y[-1] - y[0])))
        eff = len(CHECKPOINTS)
        for i, c in enumerate(curve["checkpoints"]):
            if c["accuracy"] >= 0.8:
                eff = i
                break
        sample_efficiencies.append(1 - eff / len(CHECKPOINTS))
        if len(y) > 1:
            curve_qualities.append(float(np.mean(np.diff(y) >= -0.05)))
        else:
            curve_qualities.append(0.5)

    dw = [c["difficulty"] ** 1.5 for c in all_curves]
    tw = sum(dw)
    std_score = (
        0.30 * sum(a*w for a,w in zip(asymptotic_accs, dw)) / tw +
        0.30 * sum(a*w for a,w in zip(learning_rates, dw)) / tw +
        0.20 * sum(a*w for a,w in zip(sample_efficiencies, dw)) / tw +
        0.20 * sum(a*w for a,w in zip(curve_qualities, dw)) / tw
    )

    # ═══ CONDITION B: Far-Transfer (0.50) ═══
    # Train on base system (rules + examples), then test on transfer system
    # (same structure, different surface) with rules but NO examples.
    # Measures whether learning transfers across surface-feature changes.
    transfer_scores = []
    for i, pair in enumerate(FAR_TRANSFER_PAIRS):
        base = pair["base"]
        transfer = pair["transfer"]

        with kbench.chats.new(f"far_transfer_{i}"):
            n_correct = 0
            for test_item in transfer.test_items:
                prompt_parts = [
                    f"You previously learned the rule system: **{base.name}**\n",
                    f"Description: {base.description}\n",
                    "\n**Rules:**",
                ]
                for rule in base.rules:
                    prompt_parts.append(f"- {rule}")
                prompt_parts.append(f"\n**Training examples ({min(8, len(base.examples))}):**")
                for ex in base.examples[:8]:
                    prompt_parts.append(f"  Input: {ex['input']}  →  Output: {ex['output']}")

                prompt_parts.append(f"\n\n--- NOW: NEW DOMAIN ---")
                prompt_parts.append(f"The same underlying rules apply, but in a different format.")
                prompt_parts.append(f"New system: **{transfer.name}**")
                prompt_parts.append(f"Description: {transfer.description}\n")
                prompt_parts.append("**Rules (same structure, new surface):**")
                for rule in transfer.rules:
                    prompt_parts.append(f"- {rule}")
                prompt_parts.append("\n(No examples provided for this new format — transfer your learning.)")

                test_prompt = "\n".join(prompt_parts) + (
                    f"\n\nApply the rules to this input:\n"
                    f"Input: {test_item['input']}\n\n"
                    f"Respond with ONLY a JSON object:\n"
                    f'{{"answer": "<output after applying rules>", "reasoning": "<your steps>"}}'
                )
                try:
                    result = llm.prompt(test_prompt, schema=RuleAnswer)
                    answer = result.answer
                except Exception:
                    raw = llm.prompt(test_prompt)
                    try:
                        parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
                        answer = str(parsed.get("answer", raw))
                    except Exception:
                        answer = raw
                if check_output(answer, test_item["output"]):
                    n_correct += 1

            acc = n_correct / len(transfer.test_items) if transfer.test_items else 0
            transfer_scores.append(acc)
            print(f"  Far-transfer {i} ({base.name} → {transfer.name}): {acc:.2%}")  

    far_transfer_score = np.mean(transfer_scores) if transfer_scores else 0

    # ═══ CONDITION C: Steep/Hard (0.25) ═══
    # Only 3 training examples on difficulty-3 systems
    steep_scores = []
    for system in HARD_LEARNING_SYSTEMS:
        acc_0 = _eval_system(llm, system, 0, system.test_items,
                             f"steep_{system.name}_n0")
        acc_3 = _eval_system(llm, system, 3, system.test_items,
                             f"steep_{system.name}_n3")
        # Score = how much learned from just 3 examples × accuracy
        steep_score = 0.40 * acc_3 + 0.60 * max(0, acc_3 - acc_0)
        steep_scores.append(steep_score)
        print(f"  Steep {system.name}: 0-shot={acc_0:.2%}, 3-shot={acc_3:.2%}, score={steep_score:.3f}")

    steep_score = np.mean(steep_scores) if steep_scores else 0

    # ═══ COMPOSITE ═══
    score = round(0.25 * std_score + 0.50 * far_transfer_score + 0.25 * steep_score, 4)

    # ── Logging ──
    print(f"\n{'='*60}")
    print(f"LEARNING CURVES BENCHMARK v2 RESULTS")
    print(f"{'='*60}")
    print(f"\nCondition A (Standard): {std_score:.4f}")
    print(f"Condition B (Far-Transfer): {far_transfer_score:.4f}")
    print(f"Condition C (Steep/Hard): {steep_score:.4f}")
    print(f"\nComposite (0.25*A + 0.50*B + 0.25*C): {score:.4f}")

    for curve in all_curves:
        print(f"\n--- {curve['system']} (difficulty={curve['difficulty']}) ---")
        for cp in curve["checkpoints"]:
            bar = "█" * int(cp["accuracy"] * 20)
            print(f"  n={cp['n_examples']:2d}: {cp['accuracy']:.2%} {bar}")

    return score


# ─── Run ────────────────────────────────────────────────────────────
learning_curves.run(llm=kbench.llm)
